# **Mini-Projeto Avaliativo — Análise Exploratória de Dados (Base Varejo)**

Disciplina: BI e Visualização de Dados / Aluno: Bruna Mainardes Leal / Turma: T3

Estou realizando neste notebook, a Análise Exploratória de Dados (AED) da base **Base Varejo.csv**, seguindo as etapas obrigatórias do enunciado: carga, diagnóstico de qualidade, limpeza, estatística descritiva, agrupamentos e conclusões.

# **Sprint 1 - Importação dos Dados**


In [51]:
# Importando a base Varejo diretamente do Kaggle para iniciar a análise
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "Base Varejo.csv"

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "namespaiva/base-varejo",
    file_path,
    pandas_kwargs={"sep": ";", "encoding": "utf-8"},
)

Using Colab cache for faster access to the 'base-varejo' dataset.


In [52]:
# Importando a biblioteca ultilizada
import pandas as pd

pd.set_option("display.max_columns", None)

In [53]:
# Fiz uma remoção das colunas "fantasma", que estavam 100% vazias e
# eram resultantes da estrutura original do arquivo CSV
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
print(f"Shape após remover colunas vazias: {df.shape}")
df.head()

Shape após remover colunas vazias: (830000, 10)


,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI
4,01/02/2019,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO


# **Sprint 2 e 3 - Transformação de Strings, Integer, Float e Datetime / Limpeza de Nulos e Duplicatas**


In [54]:
# Verificando os valores nulos por coluna
print("Valores nulos por coluna:")
print(df.isna().sum())

Valores nulos por coluna:
DATA         0
CO_ID        0
CL_ID        0
CL_GENERO    0
CL_EC        0
CL_FHL       0
CL_SEG       0
PR_ID        0
PR_CAT       0
PR_NOME      0
dtype: int64


In [55]:
#  Verificando as linhas duplicadas, considerando todas as colunas
n_duplicadas = df.duplicated().sum()
print(f"Linhas totalmente duplicadas: {n_duplicadas} ({n_duplicadas / len(df):.2%} da base)")

Linhas totalmente duplicadas: 96553 (11.63% da base)


In [56]:
# Verificando as inconsistências: categorias de produto não informadas ("#N/D") e datas inválidas
n_categoria_invalida = (df["PR_CAT"] == "#N/D").sum()
print(f"Registros com categoria de produto '#N/D': {n_categoria_invalida} "
      f"({n_categoria_invalida / len(df):.2%} da base)")

datas_teste = pd.to_datetime(df["DATA"], format="%d/%m/%Y", errors="coerce")
n_datas_invalidas = datas_teste.isna().sum()
print(f"Datas que não seguem o padrão dd/mm/aaaa: {n_datas_invalidas}")

Registros com categoria de produto '#N/D': 3650 (0.44% da base)
Datas que não seguem o padrão dd/mm/aaaa: 0


**Conclusão:** Identifiquei que a base não possui valores nulos comuns, como NaN, nas principais colunas. Porém, encontrei duas inconsistências: existem algumas linhas duplicadas e alguns produtos estão com a categoria marcada como #N/D, que pode ser considerado um valor nulo em formato de texto. Também verifiquei que todas as datas estão em um formato válido.



# **Sprint 3**

In [57]:
# Encontrei alguns valores marcados como #N/D na categoria dos produtos.
# Decidi remover essas linhas porque elas representam uma quantidade muito pequena da base,
# apenas 0,44% dos dados. Pensei em substituir esses valores pela categoria que mais aparece,
# mas isso poderia colocar informações que não são verdadeiras e prejudicar a análise das
# categorias mais adiante, por isso, achei melhor remover essas informações da base.
df_limpo = df[df["PR_CAT"] != "#N/D"].copy()
print(f"Linhas após remover categoria '#N/D': {len(df_limpo)}")

Linhas após remover categoria '#N/D': 826350


In [58]:
# Percebi que cada linha da base representa um item comprado dentro de uma compra.
# Como não existe uma coluna mostrando a quantidade de produtos, quando encontrei
# duas linhas exatamente iguais, entendi que provavelmente eram registros duplicados
# e não duas unidades do mesmo produto, por isso, decidi remover essas linhas duplicadas.
antes = len(df_limpo)
df_limpo = df_limpo.drop_duplicates()
print(f"Duplicatas removidas: {antes - len(df_limpo)}")
print(f"Total de registros após limpeza: {len(df_limpo)}")


Duplicatas removidas: 96131
Total de registros após limpeza: 730219


In [59]:
# Converti a coluna DATA de texto para o formato de data
df_limpo["DATA"] = pd.to_datetime(df_limpo["DATA"], format="%d/%m/%Y")

# Converti as colunas com poucas categorias para o tipo 'category'.
for col in ["CL_GENERO", "CL_SEG", "PR_CAT"]:
    df_limpo[col] = df_limpo[col].astype("category")

print(df_limpo.dtypes)

DATA         datetime64[ns]
CO_ID                 int64
CL_ID                 int64
CL_GENERO          category
CL_EC                 int64
CL_FHL                int64
CL_SEG             category
PR_ID                 int64
PR_CAT             category
PR_NOME              object
dtype: object


In [60]:
# Conferência pós-limpeza
print(f"Shape final: {df_limpo.shape}")
print(f"Nulos restantes: {df_limpo.isna().sum().sum()}")
print(f"Duplicatas restantes: {df_limpo.duplicated().sum()}")
df_limpo.head()

Shape final: (730219, 10)
Nulos restantes: 0
Duplicatas restantes: 0


,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME
0,2019-02-01,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA
1,2019-02-01,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS
2,2019-02-01,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO
3,2019-02-01,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI
4,2019-02-01,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO


# **Sprint 4 — Estatística descritiva**

In [61]:
# Analisei a coluna CL_FHL para entender melhor a quantidade de filhos dos clientes,
# calculando a média, mediana, desvio padrão, moda, maior e menor valor e a quantidade de registros.
# Depois mostrei todos esses resultados
filhos = df_limpo["CL_FHL"]

estatisticas_filhos = {
    "média": filhos.mean(),
    "mediana": filhos.median(),
    "desvio padrão": filhos.std(),
    "moda": filhos.mode()[0],
    "máximo": filhos.max(),
    "mínimo": filhos.min(),
    "contagem": filhos.count(),
}

for nome, valor in estatisticas_filhos.items():
    print(f"{nome}: {valor}")



média: 1.145979493823086
mediana: 0.0
desvio padrão: 1.4168585700403418
moda: 0
máximo: 4
mínimo: 0
contagem: 730219


# **Sprint 5 - Relatório e Documentação**

Analisei os dados de três formas: para descobrir quem mais compra,
quais categorias de produtos vendem mais e como as vendas mudam ao longo dos meses.

In [62]:
# Agrupamento 1: total de itens comprados e clientes únicos por gênero
por_genero = df_limpo.groupby("CL_GENERO", observed=True).agg(
    total_itens_comprados=("PR_ID", "count"),
    clientes_unicos=("CL_ID", "nunique"),
).sort_values("total_itens_comprados", ascending=False)

por_genero


,total_itens_comprados,clientes_unicos
CL_GENERO,,
F,380735,519
M,349484,481


In [63]:
# Agrupamento 2: total de itens vendidos por categoria de produto
por_categoria = (
    df_limpo.groupby("PR_CAT", observed=True)["PR_ID"]
    .count()
    .sort_values(ascending=False)
)

por_categoria

,PR_ID
PR_CAT,
ALIMENTOS,384197
HIGIENE,137702
LIMPEZA,128632
BEBIDAS,38264
PET,28553
ACESSORIOS,12871


In [64]:
# Agrupamento 3: total de itens vendidos por mês/ano
df_limpo["ANO_MES"] = df_limpo["DATA"].dt.to_period("M")

vendas_por_mes = pd.pivot_table(
    df_limpo, index="ANO_MES", values="PR_ID", aggfunc="count"
).rename(columns={"PR_ID": "itens_vendidos"})

vendas_por_mes.head(12)

,itens_vendidos
ANO_MES,
2019-01,6468
2019-02,16077
2019-03,9789
2019-04,16179
2019-05,28003
2019-06,19396
2019-07,10699
2019-08,15836
2019-09,16824


# **Conclusões**

1. **Base após a limpeza:** mantive a maior parte dos dados depois de remover as
duplicatas e os registros com #N/D.

2. **Perfil dos clientes:** a maioria dos clientes não possui filhos ou possui poucos filhos.

3. **Compras por gênero:** os clientes dos gêneros F e M apresentam uma quantidade de compras parecida, sem uma diferença muito grande entre os grupos.

4. **Categorias mais vendidas:** a categoria *alimentos* foi a que apresentou mais vendas, seguida por *higiene* e *limpeza*.

5. **Vendas ao longo dos meses:** percebi que a quantidade de vendas varia de um mês para outro, com alguns períodos apresentando maior movimento.

6. **Limitações da base:** como não temos informações sobre o valor das compras nem a quantidade de cada produto, não foi possível analisar o faturamento de forma mais detalhada.

# **Sprint 6 - Versionamento**

In [50]:
# Salvei a base de dados já limpa em um arquivo CSV para poder guardar e acompanhar
# as alterações do projeto no repositório
df_limpo.to_csv("df_limpo.csv", sep=";", index=False, encoding="utf-8")
print("Arquivo df_limpo.csv salvo com sucesso.")

Arquivo df_limpo.csv salvo com sucesso.
